# EXP001 — Feature × Metric Learning
**KWS Research — Colab GPU Runner**
Design baseline: `research-design-v2.0`

1. Run primary cells first (LogMel + ProtoNet × 3 seeds)
2. Then PCEN + ProtoNet
3. Then secondary (GE2E, Triplet)
4. Results saved to Drive + download zip

In [ ]:
import os
import subprocess

def run(cmd, timeout=None):
    """Run a shell command and stream output."""
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, text=True,
                           bufsize=1)
    for line in proc.stdout:
        print(line, end='')
    proc.wait()
    if proc.returncode != 0:
        print(f"\n⚠ EXIT CODE: {proc.returncode}")
    return proc.returncode

## Step 1 — Clone + Install

In [ ]:
print("=" * 60)
print("STEP 1: Clone + Install")
print("=" * 60)

REPO = "https://github.com/NgocAnLam/KWS_Research.git"
if not os.path.exists("/content/KWS_Research"):
    run(f"git clone {REPO}")
%cd /content/KWS_Research
run("pip install -e . -q")
run("pip install soundfile scikit-learn scipy thop -q")
print("\n✅ Setup complete")

## Step 2 — Download GSCv2 Dataset

In [ ]:
print("=" * 60)
print("STEP 2: Download GSCv2")
print("=" * 60)

import os
if not os.path.exists("data/raw/speech_commands_v0.02/README.md"):
    run("wget -q http://download.tensorflow.org/data/speech_commands_v0.02.tar.gz")
    run("mkdir -p data/raw/speech_commands_v0.02")
    run("tar xzf speech_commands_v0.02.tar.gz -C data/raw/speech_commands_v0.02/")
    run("rm speech_commands_v0.02.tar.gz")
    print("Dataset downloaded")
else:
    print("Dataset already exists")

# Verify
run("python -c \"from kws_framework.dataset.dataset import GSCv2Dataset; d = GSCv2Dataset('data/raw/speech_commands_v0.02'); print(f'OK: {len(d)} files')\"")
print("\n✅ Dataset ready")

## Step 3 — Pull latest + Verify GPU

In [ ]:
print("=" * 60)
print("STEP 3: Pull latest + GPU Info")
print("=" * 60)

run("git pull origin main")

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA:    {torch.version.cuda}")
print(f"GPU:     {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")
print("\n✅ GPU ready")

## Step 4 — PRIMARY: LogMel + ProtoNet (3 seeds)

In [ ]:
print("=" * 60)
print("PRIMARY: LogMel + ProtoNet")
print("=" * 60)

CFG = "src/configs/exp001_colab.yaml"

for seed in [42, 43, 44]:
    print(f"\n--- Seed {seed} ---")
    run(f"python src/scripts/run_experiment.py --config {CFG} --feature log_mel --loss prototypical --seed {seed}")

print("\n✅ LogMel + ProtoNet complete")

## Step 5 — PRIMARY: PCEN + ProtoNet (3 seeds)

In [ ]:
print("=" * 60)
print("PRIMARY: PCEN + ProtoNet")
print("=" * 60)

for seed in [42, 43, 44]:
    print(f"\n--- Seed {seed} ---")
    run(f"python src/scripts/run_experiment.py --config {CFG} --feature log_mel_pcen --loss prototypical --seed {seed}")

print("\n✅ PCEN + ProtoNet complete")

## Step 6 — SECONDARY: GE2E + Triplet

In [ ]:
print("=" * 60)
print("SECONDARY: GE2E + Triplet")
print("=" * 60)

# LogMel + secondary losses
run(f"python src/scripts/run_experiment.py --config {CFG} --feature log_mel --loss ge2e --seed 42")
run(f"python src/scripts/run_experiment.py --config {CFG} --feature log_mel --loss triplet --seed 42")

# PCEN + secondary losses (nếu còn thời gian)
# run(f"python src/scripts/run_experiment.py --config {CFG} --feature log_mel_pcen --loss ge2e --seed 42")
# run(f"python src/scripts/run_experiment.py --config {CFG} --feature log_mel_pcen --loss triplet --seed 42")

print("\n✅ Secondary complete")

## Step 7 — Generate summary + Save

In [ ]:
print("=" * 60)
print("STEP 7: Generate summary + Save to Drive")
print("=" * 60)

# Generate summary.csv
import json, os
base = "experiments/exp001"
lines = []
for d in sorted(os.listdir(base)):
    mf = os.path.join(base, d, "metrics.json")
    if os.path.exists(mf):
        m = json.load(open(mf))
        lines.append(f"{m['feature']},{m['loss']},{m['seed']},{m['accuracy']},{m['f1']},{m['auc']},{m['eer']},{m['acc_at_1pct_far']},{m['acc_at_5pct_far']},{m['total_params']},{m['inference_latency_ms']}")

with open(os.path.join(base, "summary.csv"), "w") as f:
    f.write("Feature,Loss,Seed,Accuracy,F1,AUC,EER,Acc@1%FAR,Acc@5%FAR,Params,Latency_ms\n")
    for line in lines:
        f.write(line + "\n")
print(f"Summary: {len(lines)} cells written to {base}/summary.csv")
!cat {base}/summary.csv

# Save to Google Drive
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/KWS_Research/experiments
!cp -r /content/KWS_Research/experiments/exp001 /content/drive/MyDrive/KWS_Research/experiments/exp001
print("\n✅ Saved to MyDrive/KWS_Research/experiments/exp001/")

# Download zip
!zip -r /content/exp001_results.zip /content/KWS_Research/experiments/exp001/
from google.colab import files
files.download('/content/exp001_results.zip')